In [1]:
import random
import warnings
import traceback
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import quad4fhe

from plain_experiment_io import (
    build_quad4fhe_run_record,
    make_experiment_payload,
    save_csv,
    save_json,
    tee_stdout_stderr,
    write_combined_dataset_json,
)


# ============================================================
# Global Configuration
# ============================================================
SEED = 2026
TRAIN_SIZE = 0.60
VAL_SIZE   = 0.20
TEST_SIZE  = 0.20
HIDDEN_DIMS = [64, 128, 256]
EPOCHS = 200
LR = 1e-3

EARLY_STOP_PATIENCE = 20
EARLY_STOP_MIN_DELTA = 1e-5

DIABETES_CSV_PATH = "diabetes_dataset.csv"
DIABETES_TARGET_COL = None  # auto-detect

BASELINES = ["square", "ls_poly_2", "ls_poly_3", "ls_poly_5", "ls_poly_7",
             "remez_2", "remez_3", "remez_5", "remez_7",
             "ola", "precise_a11"]

# ============================================================
# Automatic result export
# ============================================================
DATASET_NAME = "Diabetes"
EXPERIMENT_NAME = "fulltrain"
OUTPUT_DIR = Path("..") / "results" / DATASET_NAME / EXPERIMENT_NAME
LOG_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_result.txt"
JSON_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_results.json"
SUMMARY_CSV_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_summary.csv"


# ============================================================
# Decision-preservation diagnostics
# ============================================================
def _quad_report_diagnostics(result, split: str, n_expected: int = None):
    """Return calibration/test agreement and mismatch count for Quad4FHE.

    Newer quad4fhe versions expose result.fit_diagnostics/result.test_diagnostics.
    The fallback below extracts the same information from report_vs_pseudo.
    """
    attr_name = "fit_diagnostics" if split == "fit" else "test_diagnostics"
    diag = getattr(result, attr_name, None)
    if diag is not None:
        return dict(diag)

    df = getattr(result, "report_vs_pseudo", None)
    if df is None:
        return {
            "split": split, "n": n_expected, "agreement": None,
            "mismatch_count": None, "exact_preserved": None,
        }
    row = df[(df["Method"] == "Quad4FHE") & (df["Split"] == split)]
    if len(row) == 0:
        return {
            "split": split, "n": n_expected, "agreement": None,
            "mismatch_count": None, "exact_preserved": None,
        }
    agreement = float(row.iloc[0]["Agreement"])
    mismatch = None if n_expected is None else int(round((1.0 - agreement) * int(n_expected)))
    return {
        "split": split,
        "n": n_expected,
        "agreement": agreement,
        "mismatch_count": mismatch,
        "exact_preserved": None if mismatch is None else bool(mismatch == 0),
    }


def _quad_solver_diagnostics(result):
    """Return margin/slack diagnostics when the quad4fhe library exposes them."""
    return dict(getattr(result, "solver_diagnostics", None) or {})


# ============================================================
# Model
# ============================================================
class SingleHiddenLayerMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x))).squeeze(-1)


# ============================================================
# Training with early stopping
# ============================================================
class EarlyStopping:
    def __init__(self, patience=EARLY_STOP_PATIENCE, min_delta=EARLY_STOP_MIN_DELTA):
        self.patience, self.min_delta = patience, min_delta
        self.best_loss = float("inf"); self.counter = 0
        self.best_epoch = 0; self.best_state = None

    def step(self, val_loss, epoch, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss, self.counter, self.best_epoch = val_loss, 0, epoch
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


def train_relu_mlp(X_tr, y_tr, X_va, y_va, input_dim, hidden_dim):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SingleHiddenLayerMLP(input_dim, hidden_dim).to(device)

    X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
    y_tr_t = torch.tensor(y_tr, dtype=torch.float32).to(device)
    X_va_t = torch.tensor(X_va, dtype=torch.float32).to(device)
    y_va_t = torch.tensor(y_va, dtype=torch.float32).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)
    bce = nn.BCEWithLogitsLoss()
    stopper = EarlyStopping()

    print(f"  Architecture: Linear({input_dim}->{hidden_dim}) -> ReLU -> Linear({hidden_dim}->1)")
    print(f"  Device={device}, epochs={EPOCHS}, lr={LR}")

    for epoch in range(EPOCHS):
        model.train()
        opt.zero_grad()
        train_loss = bce(model(X_tr_t), y_tr_t)
        train_loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = bce(model(X_va_t), y_va_t).item()

        if (epoch + 1) % 50 == 0:
            print(f"    epoch {epoch+1:>4d}  train_loss={train_loss.item():.6f}  val_loss={val_loss:.6f}")

        if stopper.step(val_loss, epoch + 1, model):
            print(f"  Early stopping @ epoch {epoch + 1} "
                  f"(best val_loss={stopper.best_loss:.6f} @ epoch {stopper.best_epoch})")
            stopper.restore(model)
            break
    else:
        stopper.restore(model)
    model.cpu()
    model.eval()
    return model


# ============================================================
# Data loading
# ============================================================
def load_diabetes_csv(csv_path, target_col=None):
    df = pd.read_csv(csv_path)
    print(f"Loaded CSV: {csv_path}  shape={df.shape}")

    if target_col is not None and target_col in df.columns:
        tgt = target_col
    else:
        candidates = ['Outcome', 'outcome', 'Diabetes', 'diabetes', 'Target', 'target',
                       'Class', 'class', 'Label', 'label', 'Diabetic', 'diabetic']
        tgt = next((c for c in candidates if c in df.columns), df.columns[-1])
    print(f"  Target column: '{tgt}'")

    y_all = df[tgt].values.astype(int)
    feature_cols = [c for c in df.columns if c != tgt]
    X_all = df[feature_cols].values.astype(np.float64)

    # Fill NaNs with column medians
    if np.isnan(X_all).any():
        medians = np.nanmedian(X_all, axis=0)
        for j in range(X_all.shape[1]):
            mask = np.isnan(X_all[:, j])
            X_all[mask, j] = medians[j]

    # Ensure 0/1 labels
    uniq = np.unique(y_all)
    if not np.array_equal(uniq, np.array([0, 1])):
        y_all = (y_all == uniq[1]).astype(int)

    print(f"  samples={len(y_all)}, features={len(feature_cols)}, "
          f"pos={int(np.sum(y_all == 1))}, neg={int(np.sum(y_all == 0))}")
    return X_all, y_all


# ============================================================
# Main
# ============================================================
def main():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    print("=" * 78)
    print("Step 1: Load and split diabetes dataset")
    print("=" * 78)
    X_all, y_all = load_diabetes_csv(DIABETES_CSV_PATH, DIABETES_TARGET_COL)

    X_tr_raw, X_tmp, y_tr, y_tmp = train_test_split(
        X_all, y_all, test_size=(VAL_SIZE + TEST_SIZE),
        random_state=SEED, stratify=y_all)
    val_ratio = VAL_SIZE / (VAL_SIZE + TEST_SIZE)
    X_va_raw, X_te_raw, y_va, y_te = train_test_split(
        X_tmp, y_tmp, test_size=(1.0 - val_ratio),
        random_state=SEED, stratify=y_tmp)

    scaler = StandardScaler().fit(X_tr_raw)
    X_tr = scaler.transform(X_tr_raw).astype(np.float64)
    X_va = scaler.transform(X_va_raw).astype(np.float64)
    X_te = scaler.transform(X_te_raw).astype(np.float64)

    print(f"  train={len(X_tr)}, val={len(X_va)}, test={len(X_te)}, d={X_tr.shape[1]}")

    payload = make_experiment_payload(
        dataset=DATASET_NAME,
        experiment=EXPERIMENT_NAME,
        seed=SEED,
        dataset_info={
            "input_dim": int(X_tr.shape[1]),
            "num_classes": 2,
            "splits": {"train": len(X_tr), "val": len(X_va), "test": len(X_te)},
            "calibration_definition": "train+val",
        },
        config={
            "hidden_dims": HIDDEN_DIMS,
            "epochs": EPOCHS,
            "lr": LR,
            "train_size": TRAIN_SIZE,
            "val_size": VAL_SIZE,
            "test_size": TEST_SIZE,
            "baselines": BASELINES,
            "early_stop_patience": EARLY_STOP_PATIENCE,
            "early_stop_min_delta": EARLY_STOP_MIN_DELTA,
        },
        source_script=Path(__file__).name if "__file__" in globals() else "notebook",
        log_file=LOG_PATH,
    )

    all_summaries = []

    for hd in HIDDEN_DIMS:
        print("\n" + "#" * 78)
        print(f"#  RUNNING hidden_dim = {hd}")
        print("#" * 78)

        # ---- Train ----
        print(f"\n[Step 2] Train ReLU-MLP (hidden_dim={hd})")
        model = train_relu_mlp(X_tr, y_tr, X_va, y_va, X_tr.shape[1], hd)

        # ---- Replace with quad4fhe ----
        # fit_data = train + val combined.
        X_fit = np.vstack([X_tr, X_va])
        y_fit = np.concatenate([y_tr, y_va])

        print(f"\n[Step 3] quad4fhe.replace(...)  [fit on {len(X_fit)} samples]")
        result = quad4fhe.replace(
            task          = "binary",
            model         = model,
            hidden_layer  = "fc1",
            output_layer  = "fc2",
            activation    = "relu",
            fit_data      = (X_fit, y_fit),
            test_data     = (X_te, y_te),
            baselines     = BASELINES,
            export_he_dir = f"he_artifacts_h{hd}",
            use_cutting_plane = False,
            use_osqp_warmstart= False,
            verbose       = True,
            seed          = SEED,
        )

        # ---- Summary for this hidden_dim ----
        fit_diag = _quad_report_diagnostics(result, "fit", n_expected=len(X_fit))
        test_diag = _quad_report_diagnostics(result, "test", n_expected=len(X_te))
        solver_diag = _quad_solver_diagnostics(result)
        all_summaries.append({
            "hidden_dim": hd,
            "method_used": result.method_used,
            "hard_feasible": result.feasible,
            "alpha": result.alpha,
            "beta":  result.beta,
            "eta":   result.eta,
            "threshold": result.threshold,
            "empirical_margin": result.empirical_margin,
            "normalized_margin": result.normalized_margin,
            "quant_decimals": result.quant_decimals,
            "calib_n": fit_diag.get("n"),
            "calib_agreement": fit_diag.get("agreement"),
            "calib_mismatch_count": fit_diag.get("mismatch_count"),
            "exact_preserved_on_calib": fit_diag.get("exact_preserved"),
            "test_agreement": test_diag.get("agreement"),
            "test_mismatch_count": test_diag.get("mismatch_count"),
            "calib_test_agreement_gap": (None if fit_diag.get("agreement") is None or test_diag.get("agreement") is None
                                         else fit_diag.get("agreement") - test_diag.get("agreement")),
            "slack_positive_count": solver_diag.get("slack_positive_count"),
            "sum_slack": solver_diag.get("sum_slack"),
            "max_slack": solver_diag.get("max_slack"),
            "constraint_version": getattr(result, "constraint_version", None),
        })
        payload["runs"].append(build_quad4fhe_run_record(
            result=result,
            key=f"hidden_dim={hd}",
            hidden_dim=hd,
            fit_n=len(X_fit),
            test_n=len(X_te),
            pool_fraction=None,
            rep_fit_size=None,
        ))
        # Save a checkpoint after each hidden dimension, so a long experiment
        # still leaves a usable JSON if a later run fails.
        payload["cross_hidden_dim_summary"] = all_summaries
        save_json(payload, JSON_PATH)
        save_csv(all_summaries, SUMMARY_CSV_PATH)

        print(f"  calibration agreement={fit_diag.get('agreement')} "
              f"mismatches={fit_diag.get('mismatch_count')} "
              f"exact_preserved={fit_diag.get('exact_preserved')}")
        print(f"  test agreement={test_diag.get('agreement')} "
              f"mismatches={test_diag.get('mismatch_count')}")

        # ---- Drop-in verification ----
        rm = result.replaced_model.eval()
        with torch.no_grad():
            logits_quad = rm(torch.tensor(X_te, dtype=torch.float32)).cpu().numpy()
        preds_quad = (logits_quad > result.threshold).astype(int)
        print(f"\n  Quad4FHE test accuracy = {np.mean(preds_quad == y_te):.4f}")

    # ---- Cross-hidden-dim summary ----
    print("\n" + "=" * 78)
    print("Cross-hidden-dim summary")
    print("=" * 78)
    df = pd.DataFrame(all_summaries)
    with pd.option_context("display.float_format", "{:.6f}".format):
        print(df.to_string(index=False))

    payload["cross_hidden_dim_summary"] = all_summaries
    save_json(payload, JSON_PATH)
    save_csv(all_summaries, SUMMARY_CSV_PATH)
    write_combined_dataset_json(DATASET_NAME, root_dir=OUTPUT_DIR.parent.parent)

    print("\n" + "=" * 78)
    print("Done.")
    print("=" * 78)


if __name__ == "__main__":
    try:
        with tee_stdout_stderr(LOG_PATH):
            main()
    except Exception:
        # The traceback is written both to screen and to the autosaved log.
        traceback.print_exc()
        raise

[autosave] stdout/stderr log -> ..\results\Diabetes\fulltrain\Diabetes_fulltrain_result.txt
Step 1: Load and split diabetes dataset
Loaded CSV: diabetes_dataset.csv  shape=(9538, 17)
  Target column: 'Outcome'
  samples=9538, features=16, pos=3282, neg=6256
  train=5722, val=1908, test=1908, d=16

##############################################################################
#  RUNNING hidden_dim = 64
##############################################################################

[Step 2] Train ReLU-MLP (hidden_dim=64)
  Architecture: Linear(16->64) -> ReLU -> Linear(64->1)
  Device=cuda, epochs=200, lr=0.001
    epoch   50  train_loss=0.511190  val_loss=0.509740
    epoch  100  train_loss=0.332608  val_loss=0.332707
    epoch  150  train_loss=0.189416  val_loss=0.191762
    epoch  200  train_loss=0.118981  val_loss=0.121886

[Step 3] quad4fhe.replace(...)  [fit on 7630 samples]

Step 1: Extract folded model weights
hidden_layer: Linear (in=16, out=64)
  found activation ReLU at 'act' 